In [0]:
selected_season="2023/2024"   #Write the season that you want with the following format "2021/2022"
selected_month=7

email_to_send=['florian.girardi-ext@ldc.com'] #list of emails the report will be sent to, format: ['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com','nicolas.benaicha@ldc.com']


In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import re
import plotly.io as pio
url = "https://ldcom365.sharepoint.com"



In [0]:
history=sp_mgr.read_pd_from_excel('/sites/GRP-TradingLineups/Lineups/Merged/history and lineups.xlsx')


In [0]:
mask_corn = (history['Product'] == 'Wheat')

history['Month']=history['Date'].dt.month

history['Region'] = history['Region'].fillna('UNKNOWN')
history['Destination'] = history['Destination'].fillna('UNKNOWN')

# Define the season logic

def assign_season_barley(row):
    year = row['Year']
    month = row['Month']


    if pd.isna(year) or pd.isna(month):
        return None  # or "Unknown"

    year = int(year)
    month = int(month)

    if month >= 12:
        season_start = year
    else:
        season_start = year - 1

    return f"{season_start}/{season_start + 1}"

# Apply to create the new column
history['Season'] = history.apply(assign_season_barley,axis=1)

# Define today's date
today = pd.Timestamp.today()


## CURRENT MONTH

In [0]:
# Step 3: Filter the DataFrame
filtered_df = history[(history['Season'] == selected_season) & (history['Date'].dt.month == selected_month)]

filtered_df_W=filtered_df[filtered_df['Product']=='Wheat']
filtered_df_W['Date'] = filtered_df_W['Date'].dt.date

### BY REGION

In [0]:
# Group sailed quantities by region
sailed = filtered_df_W[filtered_df_W['Status'] == 'SAILED'].groupby('Region', as_index=False)['Quantity'].sum()
sailed = sailed.rename(columns={'Quantity': 'Quantity_Sailed'})

# Group announced quantities by region
announced = filtered_df_W[filtered_df_W['Status'] == 'ANNOUNCED'].groupby('Region', as_index=False)['Quantity'].sum()
announced = announced.rename(columns={'Quantity': 'Quantity_Announced'})

# Merge both
region_totals = pd.merge(sailed, announced, on='Region', how='outer').fillna(0)

# Add total column
region_totals['Total'] = region_totals['Quantity_Sailed'] + region_totals['Quantity_Announced']

# Optional: sort by total
region_totals = region_totals.sort_values(by='Total', ascending=False)

filtered_df_year = history[(history['Season']==selected_season)]
filtered_df_W_year=filtered_df_year[filtered_df_year['Product']=='Wheat']
filtered_df_W_year=filtered_df_W_year[filtered_df_W_year['Origin']=='ARG']


# Ensure Date is datetime type if not already
filtered_df_W_year['Date'] = pd.to_datetime(filtered_df_W_year['Date'])

# Add Month column (numerical)
filtered_df_W_year['Month'] = filtered_df_W_year['Date'].dt.month

# Clean country names if needed
filtered_df_W_year['Destination'] = filtered_df_W_year['Destination'].str.upper().str.strip()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table = (
    filtered_df_W_year
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
     # Ensure months go 1 to 12
)

# Optional: round values
monthly_table = monthly_table.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table.columns.name = None
monthly_table = monthly_table.reset_index()


# Clean country names if needed
filtered_df_W_year['Destination'] = filtered_df_W_year['Destination'].str.upper().str.strip()

filtered_df_W_year_sailed=filtered_df_W_year[filtered_df_W_year['Status']=='SAILED']
filtered_df_W_year_anc=filtered_df_W_year[filtered_df_W_year['Status']=='ANNOUNCED']

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_sailed = (
    filtered_df_W_year_sailed
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
         # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_sailed = monthly_table_sailed.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_sailed.columns.name = None
monthly_table_sailed = monthly_table_sailed.reset_index()

# Pivot table: Country (Destination) x Month with Quantity summed
monthly_table_ancd = (
    filtered_df_W_year_anc
    .groupby(['Destination', 'Month'])['Quantity']
    .sum()
    .unstack(fill_value=0)  # Fill months with 0 if no shipments
   # Ensure months go 1 to 12
)

# Optional: round values
monthly_table_ancd = monthly_table_ancd.round(3)

# If you want to reset index and rename columns to be like 1, 2, 3...:
monthly_table_ancd.columns.name = None
monthly_table_ancd = monthly_table_ancd.reset_index()

from datetime import datetime

# Step 1: Add 'Month' column if missing
filtered_df_W_year['Month'] = filtered_df_W_year['Date'].dt.month

# Step 2: Get current month + 1
cutoff_month = (selected_month + 1)

# Step 3: Filter up to current month + 1
filtered_df_W_cut = filtered_df_W_year[filtered_df_W_year['Month'] <= cutoff_month]

# Step 4: Create pivot table by Region and Month
quantity_by_region_month = pd.pivot_table(
    filtered_df_W_cut,
    values='Quantity',
    index='Region',
    columns='Month',
    aggfunc='sum',
    fill_value=0
)

# Step 5: Replace month numbers with names (Jan, Feb, etc.)
quantity_by_region_month.columns = [datetime(1900, m, 1).strftime('%b') for m in quantity_by_region_month.columns]


# Step 2: Extract year and month of that reference date

ref_month = selected_month

history_W=history[history['Product']=='Wheat']

# Step 3: Filter the DataFrame
history_W = history_W[(history_W['Date'].dt.month == ref_month)]

sailed_df = history_W[history_W['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()


# Define desired column order for marketing year starting in December
marketing_months = [12, 1, 2,3, 4, 5, 6, 7, 8, 9, 10,11]

# Function to reorder columns
def reorder_months(df):
    available_months = [m for m in marketing_months if m in df.columns]
    return df[['Destination'] + available_months]

# Apply to your DataFrames
monthly_table = reorder_months(monthly_table)
monthly_table_sailed = reorder_months(monthly_table_sailed)
monthly_table_ancd = reorder_months(monthly_table_ancd)



## NEED TO CONNECT TO MARS TO REPLACE THE FORECASTS FROM OSA WITH THE ONES FORM MARS

### Extract from Mars

In [0]:

country_mapping = {
    "Afghanistan": "Afghanistan",
    "Albania": "Albania",
    "Algeria": "Algeria",
    "Angola": "Angola",
    "Armenia": "Armenia",
    "Australia": "Australia",
    "Austria": "Austria",
    "Azerbaijan": "Azerbaijan",
    "Bahrain": "Bahrain",
    "Bangladesh": "Bangladesh",
    "Barbados": "Barbados",
    "Belarus": "Belarus",
    "Belgium&Luxembourg": "Belgium",  # Split into two countries
    "Bhutan": "Bhutan",
    "Bolivia, Plurinational State of":"Bolivia",
    "Bosnia and Herzegovina": "Bosnia and Herzegovina",
    "Botswana": "Botswana",
    "Brazil": "Brazil",
    "Brunei Darussalam": "Brunei",
    "Burkina Faso": "Burkina Faso",
    "Burundi":"Burundi",
    "Cambodia": "Cambodia",
    "Cameroon": "Cameroon",
    "Canada": "Canada",
    "Cape Verde": "Cabo Verde",
    "Chile": "Chile",
    "China": "China",
    "Colombia": "Colombia",
    "Congo": "Republic of the Congo",
    "Congo, the Democratic Republic of the": "Democratic Republic of Congo",
    "Cook Islands": "Cook Islands",
    "Costa Rica": "Costa Rica",
    "Croatia": "Croatia",
    "Cuba": "Cuba",
    "Cyprus": "Cyprus",
    "Czech Republic": "Czechia",
    "Côte d'Ivoire": "Ivory Coast",
    "Dominican Republic": "Dominican Republic",
    "Ecuador": "Ecuador",
    "Egypt": "Egypt",
    "El Salvador": "El Salvador",
    "Estonia": "Estonia",
    "Fiji": "Fiji",
    "Finland": "Finland",
    "France": "France",
    "French Polynesia": "French Polynesia",
    "Georgia": "Georgia",
    "Germany": "Germany",
    "Ghana": "Ghana",
    "Greece": "Greece",
    "Guatemala": "Guatemala",
    "Guinea": "Guinea",
    "Guyana": "Guyana",
    "Haiti": "Haiti",
    "Honduras": "Honduras",
    "Hong Kong": "Hong Kong",
    "Hungary": "Hungary",
    "Iceland": "Iceland",
    "India": "India",
    "Indonesia": "Indonesia",
    "Iran, Islamic Republic of": "Iran",
    "Iraq": "Iraq",
    "Ireland": "Ireland",
    "Israel": "Israel",
    "Italy": "Italy",
    "Jamaica": "Jamaica",
    "Japan": "Japan",
    "Jordan": "Jordan",
    "Kazakhstan": "Kazakhstan",
    "Korea, Republic of": "South Korea",
    "Kosovo": "Kosovo",
    "Kuwait": "Kuwait",
    "Lao People's Democratic Republic": "Laos",
    "Latvia": "Latvia",
    "Lebanon": "Lebanon",
    "Lesotho": "Lesotho",
    "Liberia": "Liberia",
    "Libya Arab Jamahiriya": "Libya",
    "Lithuania": "Lithuania",
    "Macao": "Macau",
    "Macedonia, the former Yugoslav Republic of": "North Macedonia",
    "Malaysia": "Malaysia",
    "Mali": "Mali",
    "Malta": "Malta",
    "Marshall Islands": "Marshall Islands",
    "Mauritania": "Mauritania",
    "Mauritius": "Mauritius",
    "Mexico": "Mexico",
    "Micronesia, Federated States of": "Micronesia",
    "Moldova, Republic of": "Moldova",
    "Mongolia": "Mongolia",
    "Montenegro": "Montenegro",
    "Morocco": "Morocco",
    "Mozambique": "Mozambique",
    "Myanmar": "Myanmar",
    "Namibia": "Namibia",
    "Nauru": "Nauru",
    "Nepal": "Nepal",
    "Netherlands": "Netherlands",
    "New Zealand": "New Zealand",
    "Nicaragua": "Nicaragua",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Norway": "Norway",
    "Oman": "Oman",
    "Pakistan": "Pakistan",
    "Panama": "Panama",
    "Papua New Guinea": "Papua New Guinea",
    "Peru": "Peru",
    "Philippines": "Philippines",
    "Poland": "Poland",
    "Portugal": "Portugal",
    "Puerto Rico": "Puerto Rico",
    "Qatar": "Qatar",
    "Romania": "Romania",
    "Russian Federation": "Russia",
    "Saint Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Samoa": "Samoa",
    "Saudi Arabia": "Saudi Arabia",
    "Senegal": "Senegal",
    "Serbia": "Serbia",
    "Seychelles": "Seychelles",
    "Singapore": "Singapore",
    "Slovakia": "Slovakia",
    "Slovenia": "Slovenia",
    "South Africa": "South Africa",
    "Spain": "Spain",
    "Sri Lanka": "Sri Lanka",
    "Sudan": "Sudan",
    "Swaziland": "Eswatini",
    "Sweden": "Sweden",
    "Switzerland": "Switzerland",
    "Syrian Arab Republic": "Syria",
    "Taiwan, Province of China": "Taiwan",
    "Tajikistan": "Tajikistan",
    "Thailand": "Thailand",
    "Timor-Leste": "East Timor",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "Tunisia": "Tunisia",
    "Turkey": "Turkey",
    "UNKNOWN": "UNKNOWN",  # Not a valid country
    "United Arab Emirates": "UAE",
    "United Kingdom": "United Kingdom",
    "United States": "United States",
    "Uruguay": "Uruguay",
    "Uzbekistan": "Uzbekistan",
    "Venezuela, Bolivarian Republic of": "Venezuela",
    "Viet Nam": "Vietnam",
    "Yemen": "Yemen",
    "Zimbabwe": "Zimbabwe",
    "WORLD": "WORLD",
    "Gambia":"Gambia",
    "Kenya":"Kenya",
    "Madagascar":"Madagascar",
    "Rwanda":"Rwanda",
    "Tanzania, United Republic of":"Tanzania",
    "Togo":"Togo",
    "Uganda":"Uganda",
    "S. ARABIA":"SAUDI ARABIA","SAUDI ARABI":"S. ARABIA",
    "ZZZ_Unknown":"UNKNOWN",
    "":"UNKNOWN",
    np.NaN:"UNKNOWN",
    0:"UNKNOWN",
    }


country_region_mapping = {
    "ALGERIA": "AFRICA",
    'ALEGERIA':'AFRICA',
    "ARGENTINA": "SOUTH AMERICA",
    "BANGLADESH": "ASIA",
    "BELGIUM": "EU",
    "BRAZIL": "SOUTH AMERICA",
    "CHILE": "SOUTH AMERICA",
    "CHINA": "ASIA",
    "COLOMBIA": "SOUTH AMERICA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CUBA": "CENTRAL AMERICA",
    "CYPRUS": "EU",
    "CYPRUS/GREECE": "EU",
    "DENMARK": "EU",
    "EGYPT": "AFRICA",
    "FRANCE": "EU",
    "GERMANY": "EU",
    "GREECE": "EU",
    "EU-28":"EU",
    "GREECE/ISRAEL": "EU",
    "GREECE/ITALY": "EU",
    "HOLLAND": "EU",
    "HOLLAND/GERMANY": "EU",
    "HONDURAS": "CENTRAL AMERICA",
    "INDONESIA": "SE ASIA",
    "IRAN": "ME ASIA",
    "IRELAND": "EU",
    "ISRAEL": "ME ASIA",
    "ISRAEL/GREECE": "EU",
    "ITALY": "EU",
    "IVORY COAST": "AFRICA",
    "JAPAN": "ASIA",
    "JORDAN": "ME ASIA",
    "KENYA": "AFRICA",
    "KOREA": "ASIA",
    "LEBANON": "ME ASIA",
    "LITHUANIA": "EU",
    "MALAYSIA": "SE ASIA",
    "MAURITIUS": "AFRICA",
    "MEXICO": "CENTRAL AMERICA",
    "MOROCCO": "AFRICA",
    "NETHERLANDS": "EU",
    "NETHERLANDS/GERMANY": "EU",
    "NIGERIA": "AFRICA",
    "PANAMA": "CENTRAL AMERICA",
    "PERU": "SOUTH AMERICA",
    "PHILIPPINES": "SE ASIA",
    "PORTUGAL": "EU",
    "PUERTO RICO": "CENTRAL AMERICA",
    "ROMANIA": "EU",
    "RUSSIA": "ASIA",
    "SAF": "AFRICA",
    "SAUDI ARABIA": "ME ASIA",
    "SENEGAL": "AFRICA",
    "SOUTH AFRICA": "AFRICA",
    "SOUTH KOREA": "ASIA",
    "SPAIN": "EU",
    "SYRIA": "ME ASIA",
    "TAIWAN": "ASIA",
    "TBC": "TBC",
    "THAILAND": "SE ASIA",
    "TUNISIA": "AFRICA",
    "TURKEY": "ME ASIA",
    "U.ARAB EMIRAT": "ME ASIA",
    "UAE": "ME ASIA",
    "UK": "EU",
    "UNITED KINGDOM": "EU",
    "POLAND": "EU",
    "SLOVENIA": "EU",
    "LITUANIA": "EU",
    "CROATIA": "EU",
    "GREECE + CYPRUS":'EU',

    "UNITED ARAB EMIRATES": "ME ASIA",
    "URUGUAY": "SOUTH AMERICA",
    "USA": "NORTH AMERICA",
    "UNITED STATES": "NORTH AMERICA",
    "VENEZUELA": "SOUTH AMERICA",
    "VIETNAM": "SE ASIA",
    "YEMEN": "ME ASIA",
    "": "NOT AVAILABLE",
    "MOZAMBIQUE": "AFRICA",
    "GUATEMALA": "CENTRAL AMERICA",
    "US": "NORTH AMERICA",
    "Z. OTHER EAST AFRICA": "AFRICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "AUSTRALIA": "OCEANIA",
    "NEW ZEALAND": "OCEANIA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "NICARAGUA": "CENTRAL AMERICA",
    "OMAN": "ME ASIA",
    "KUWAIT": "ME ASIA",
    "SWITZERLAND": "EU",
    "IRAQ": "ME ASIA",
    "HAITI": "CENTRAL AMERICA",
    "CANADA": "NORTH AMERICA",
    "TRINIDAD": "CENTRAL AMERICA",
    "JAMAICA": "CENTRAL AMERICA",
    "ANGOLA": "AFRICA",
    "Z. OTHER FSU": "EUROPE",
    "NORWAY": "EU",
    "NOT AVAILABLE": "",
    "ECUADOR":"SOUTH AMERICA",
    'GUYANA':"SOUTH AMERICA",
    'TANZANIA':'AFRICA',
    'LUANDA':'AFRICA',
    'LIBYA':'AFRICA', 
    'PHILIPPINNES':'SE ASIA',
    'REUNION ISLAND':'AFRICA',
    'GHANA':'AFRICA',
    'CONGO':'AFRICA',
    'NAMIBIA':'AFRICA',
    'CAPE VERDE':'AFRICA',
    'MAURITANA':'AFRICA',
    'UGANDA':'AFRICA',
    'ZIMBABWE':'AFRICA',
    'MALI':'AFRICA',
    'SUDAN':'AFRICA',
    'MAURITANIA':'AFRICA', 
    'ETHIOPIA':'AFRICA',
    'RUANDA':'AFRICA',
    'RWANDA':'AFRICA',
    'BURUNDI':'AFRICA', 
    'LYBIA':'AFRICA',
    'MAURITUS IS':'AFRICA',
    'LATVIA':'EU',
    'ESTONIA':'EU',

    'CAMEROON':'AFRICA',
    'IVORY COST':'AFRICA',
    'REUNION':'AFRICA',
    'DOM. REP':'CENTRAL AMERICA',
    'DOM REP.':'CENTRAL AMERICA',
    'DOM REP;':'CENTRAL AMERICA',
    'DOM. REP.':'CENTRAL AMERICA',
    'DOM. REP;':'CENTRAL AMERICA',
    'DOM REP':'CENTRAL AMERICA',
    'TRINIDAD & TOBAGO':'CENTRAL AMERICA',
    'GEORGIA':'ME ASIA',
    'KUWEIT':'ME ASIA',
    'BAHREIN':'ME ASIA',
    'DJBOUTI':'ME ASIA',
    'SAUDI ARABIA ':'ME ASIA',
    "Democratic Republic of Congo":"AFRICA", 	
    
    'BRUNEI':'SE ASIA',
    'MYANMAR':'SE ASIA',
    'MALAYSIA':'SE ASIA',
    'Malaysia':'SE ASIA',
    'PHILIPINES':'SE ASIA',
    'INDIA':'ASIA',
    'BELARUS':'ASIA',

    'NEW ZELAND':'OCEANIA',
    'MAURITUIS':'AFRICA',
    'U.A.E.':'ME ASIA',
    'EAU':'ME ASIA',
    'LEBANNON':'ME ASIA',
    'HOLANDA':'EU',
    'PAKISTAN':'ASIA',
    'PAKISTAN ':'ASIA',
    'RUSSIAN FEDERATION':'ASIA',
    'UNITED STATES':'ASIA',
    'TURKEY ':'ME ASIA',

    'SOUTH KOREA ':'ASIA',
    'JAPAN  ':'ASIA',
    'MALAYSIA  ':'SE ASIA',
    'IRAK':'ME ASIA', 
    'BRASIL':'SOUTH AMERICA',
    'MARRUECOS':'AFRICA',
    'ECUADOR ':'SOUTH AMERICA',
    'PARAGUAY':'SOUTH AMERICA',
    'BRAZIL ':'SOUTH AMERICA',
    'CHILE ':'SOUTH AMERICA',
    'BOLIVIA':'SOUTH AMERICA',
    'MADAGASCAR':'AFRICA',
    'GABON':'AFRICA',
    'SENEGAL ':'AFRICA',
    'GAMBIA':'AFRICA',
    'QATAR':'ME ASIA', 
    'BAHRAIN':'ME ASIA',
    'LEBANON ' :'ME ASIA',
    'BURKINA FASO':'AFRICA',
    'ALGERIA ':'AFRICA',
    'MALAWI':'AFRICA',
    'GUINEA':'AFRICA',
    'TOGO':'AFRICA',
    'LIBERIA':'AFRICA',
    'DJIBOUTI':'AFRICA',
    'VIETNAM ':'SE ASIA',
    "OTH_AFR":'AFRICA',
    "OTH_AMER":"SOUTH AMERICA",
    "OTH_EME":"ME ASIA",
    "OTH_ASIA":"SE ASIA",
    "S. ARABIA":"ME ASIA",
    "WORLD":"WORLD",
    "UNKNOWN":'UNKNOWN',
    "UNITED ARAB EMIRATES + OMAN + SAUDI ARABIA":"ME ASIA",
    "ZZZ_Unknown_Destination":'UNKNOWN',
    'KENYA+UGANDA':'AFRICA'	
    }

country_region_mapping = {
    "ALGERIA": "AFRICA",
    "ANGOLA": "AFRICA",
    "BOTSWANA": "AFRICA",
    "BURKINA FASO": "AFRICA",
    "CAMEROON": "AFRICA",
    "CAPE VERDE": "AFRICA",
    "CONGO": "AFRICA",
    "DEMOCRATIC REPUBLIC OF CONGO": "AFRICA",
    "GHANA": "AFRICA",
    "GUINEA": "AFRICA",
    "KENYA":"AFRICA",
    "UGANDA":"AFRICA",
    "IVORY COAST": "AFRICA",
    "LESOTHO": "AFRICA",
    "LIBERIA": "AFRICA",
    "MALI": "AFRICA",
    "MAURITANIA": "AFRICA",
    "MAURITIUS": "AFRICA",
    "MOROCCO": "AFRICA",
    "MOZAMBIQUE": "AFRICA",
    "NAMIBIA": "AFRICA",
    "NIGER": "AFRICA",
    "NIGERIA": "AFRICA",
    "SENEGAL": "AFRICA",
    "SEYCHELLES": "AFRICA",
    "SOUTH AFRICA": "AFRICA",
    "SUDAN": "AFRICA",
    "SWAZILAND": "AFRICA",
    "TUNISIA": "AFRICA",
    "ZIMBABWE": "AFRICA",
    "AFGHANISTAN": "ASIA",
    "BANGLADESH": "ASIA",
    "BHUTAN": "ASIA",
    "BRUNEI": "ASIA",
    "CAMBODIA": "ASIA",
    "CHINA": "ASIA",
    "HONG KONG": "ASIA",
    "INDIA": "ASIA",
    "INDONESIA": "ASIA",
    "JAPAN": "ASIA",
    "LAOS": "ASIA",
    "MACAO": "ASIA",
    "MALAYSIA": "ASIA",
    "MONGOLIA": "ASIA",
    "MYANMAR": "ASIA",
    "NEPAL": "ASIA",
    "PAKISTAN": "ASIA",
    "PHILIPPINES": "ASIA",
    "SINGAPORE": "ASIA",
    "SOUTH KOREA": "ASIA",
    "SRI LANKA": "ASIA",
    "TAIWAN": "ASIA",
    "THAILAND": "ASIA",
    "TIMOR-LESTE": "ASIA",
    "VIETNAM": "ASIA",
    "BARBADOS": "CENTRAL AMERICA",
    "COSTA RICA": "CENTRAL AMERICA",
    "CUBA": "CENTRAL AMERICA",
    "DOMINICAN REPUBLIC": "CENTRAL AMERICA",
    "EL SALVADOR": "CENTRAL AMERICA",
    "GUATEMALA": "CENTRAL AMERICA",
    "HAITI": "CENTRAL AMERICA",
    "HONDURAS": "CENTRAL AMERICA",
    "JAMAICA": "CENTRAL AMERICA",
    "NICARAGUA": "CENTRAL AMERICA",
    "PANAMA": "CENTRAL AMERICA",
    "PUERTO RICO": "CENTRAL AMERICA",
    "SAINT VINCENT": "CENTRAL AMERICA",
    "TRINIDAD AND TOBAGO": "CENTRAL AMERICA",
    "AUSTRIA": "EU",
    "BELGIUM&LUX": "EU",
    "CROATIA": "EU",
    "CYPRUS": "EU",
    "CZECH REPUBLIC": "EU",
    "ESTONIA": "EU",
    "FINLAND": "EU",
    "FRANCE": "EU",
    "GERMANY": "EU",
    "GREECE": "EU",
    "HUNGARY": "EU",
    "IRELAND": "EU",
    "ITALY": "EU",
    "LATVIA": "EU",
    "LITHUANIA": "EU",
    "MALTA": "EU",
    "NETHERLANDS": "EU",
    "POLAND": "EU",
    "PORTUGAL": "EU",
    "ROMANIA": "EU",
    "SLOVAKIA": "EU",
    "SLOVENIA": "EU",
    "SPAIN": "EU",
    "SWEDEN": "EU",
    "ALBANIA": "EUROPE NON-EU",
    "BOSNIA AND HERZEGOVINA": "EUROPE NON-EU",
    "ICELAND": "EUROPE NON-EU",
    "KOSOVO": "EUROPE NON-EU",
    "MACEDONIA": "EUROPE NON-EU",
    "MONTENEGRO": "EUROPE NON-EU",
    "NORWAY": "EUROPE NON-EU",
    "SERBIA": "EUROPE NON-EU",
    "SWITZERLAND": "EUROPE NON-EU",
    "UNITED KINGDOM": "EUROPE NON-EU",
    "ARMENIA": "FSU",
    "AZERBAIJAN": "FSU",
    "BELARUS": "FSU",
    "GEORGIA": "FSU",
    "KAZAKHSTAN": "FSU",
    "MOLDOVA": "FSU",
    "RUSSIA": "FSU",
    "TAJIKISTAN": "FSU",
    "UZBEKISTAN": "FSU",
    "BAHRAIN": "MIDDLE EAST",
    "EGYPT": "MIDDLE EAST",
    "IRAN": "MIDDLE EAST",
    "IRAQ": "MIDDLE EAST",
    "ISRAEL": "MIDDLE EAST",
    "JORDAN": "MIDDLE EAST",
    "KUWAIT": "MIDDLE EAST",
    "LEBANON": "MIDDLE EAST",
    "LIBYA": "MIDDLE EAST",
    "OMAN": "MIDDLE EAST",
    "QATAR": "MIDDLE EAST",
    "SAUDI ARABIA": "MIDDLE EAST",
    "SYRIA": "MIDDLE EAST",
    "TURKEY": "MIDDLE EAST",
    "UAE": "MIDDLE EAST",
    "YEMEN": "MIDDLE EAST",
    "CANADA": "NORTH AMERICA",
    "MEXICO": "NORTH AMERICA",
    "UNITED STATES": "NORTH AMERICA",
    "AUSTRALIA": "OCEANIA",
    "COOK ISLANDS": "OCEANIA",
    "FIJI": "OCEANIA",
    "FRENCH POLYNESIA": "OCEANIA",
    "MARSHALL ISLANDS": "OCEANIA",
    "MICRONESIA": "OCEANIA",
    "NAURU": "OCEANIA",
    "NEW ZEALAND": "OCEANIA",
    "PAPUA NEW GUINEA": "OCEANIA",
    "SAMOA": "OCEANIA",
    "BRAZIL": "SOUTH AMERICA",
    "CHILE": "SOUTH AMERICA",
    "COLOMBIA": "SOUTH AMERICA",
    "ECUADOR": "SOUTH AMERICA",
    "GUYANA": "SOUTH AMERICA",
    "PERU": "SOUTH AMERICA",
    "URUGUAY": "SOUTH AMERICA",
    "VENEZUELA": "SOUTH AMERICA",
    "OTH_AFR":'AFRICA',
    "OTH_AMER":"SOUTH AMERICA",
    "OTH_EME":"MIDDLE EAST",
    "OTH_ASIA":"SE ASIA",
    "S. ARABIA":"ME ASIA",
    "UNITED ARAB EMIRATES + OMAN + SAUDI ARABIA":"MIDDLE EAST",
    "ZZZ_Unknown_Destination":'UNKNOWN',
    'KENYA+UGANDA':'AFRICA',
    "WORLD": "WORLD",
    "UNKNOWN": "UNKNOWN"
}

In [0]:
current_season=selected_season

In [0]:
## Import LDC libs
from LDCDataAccessLayerPy import databricks_init, SharePointManager, SqlManager

# Initialize allowing me to access Sharepoint
databricks_init(dbutils, "GO")


db = SqlManager()


# {"at price", "residual"}
# TYPE = "At Price"
# TYPE = "At price"
TYPE = "Residual"

# Countries to graph
want_countries = {"Argentina"}#, "United States"}
cropyearstart_corn = {'United States': 9, 'Ukraine': 10, 'Brazil': 2, 'Argentina': 3}
cropyearstart_wheat = {'United States': 9, 'Ukraine': 10, 'Brazil': 2, 'Argentina': 12}
cropyearstart_barley = {'United States': 9, 'Ukraine': 10, 'Brazil': 2, 'Argentina': 12}


## Colors

# LDC colors
navy = "#32556E"
green = "#599536"
lightblue = "#97B8DB"
purple = "#5D465C"
lowablue = "#b8c1ff"




# Sets of colors
# [pale, saturated, dark]
redset = ["#ffadad", "#7e0000", "#ff0000"]
blueset = ["#c7cfff", "#000e65", "#0023ff"]
greenset = ["#a2cc95", "#1a531e", "#00b712"]
brownset = ["#ffddb9", "#3ca947", "#765a2d"] 

colorset = [redset, blueset, greenset, brownset]


# Set matrix to look for at-price vs residual
exp_val = ["At-price", True]
if (TYPE.title() == "Residual".title()):
  exp_val[0] = "Expected"


#### Read in from sql database

tradeflow_read = db.sql_read('MarsGrainsReplica', 
                        table = 'dbo.TradeFlow')


#### Sql keys to codes

country_id = db.sql_read('MarsGrainsReplica',table = 'dbo.Country')
commodity_id = db.sql_read('MarsGrainsReplica', table = 'dbo.Commodity')
commodity_quality_id = db.sql_read('MarsGrainsReplica', table = 'dbo.CommodityQuality')
commodity_sub_quality_id = db.sql_read('MarsGrainsReplica', table = 'dbo.CommoditySubQuality')
status_id = db.sql_read('MarsGrainsReplica', table = 'dbo.Status')

#### Rename key dfs to match the bs read in

# Country
country_id["Country"] = country_id["Name"]
country_id["country_id"] = country_id["Id"]

# Commodity
commodity_id["Commodity"] = commodity_id["Name"]
commodity_id["commodity_id"] = commodity_id["Id"]

# Quality
commodity_quality_id["CommodityQuality"] = commodity_quality_id["Name"]
commodity_quality_id["quality_id"] = commodity_quality_id["Id"]
commodity_quality_id["commodity_id"] = commodity_quality_id["Commodity"]

# Subquality
commodity_sub_quality_id["CommoditySubQuality"] = commodity_sub_quality_id["Name"]
commodity_sub_quality_id["subquality_id"] = commodity_sub_quality_id["Id"]
commodity_sub_quality_id["quality_id"] = commodity_sub_quality_id["CommodityQuality"]

# Get rid of extra columns to prevent column name problems
country_id = country_id[["country_id", "Country"]]
commodity_id = commodity_id[["Commodity", "commodity_id"]]
commodity_quality_id = commodity_quality_id[["commodity_id", "quality_id", "CommodityQuality"]]
commodity_sub_quality_id = commodity_sub_quality_id[["CommoditySubQuality", "subquality_id", "quality_id"]]


#### Map IDs

#### Get rid of unmapped countries

# Init
tradeflow = tradeflow_read.copy(deep = True)

# Rename b/s df to match the ids
tradeflow = tradeflow.rename({"SourceCountry": "country_id",            
                "Commodity": "commodity_id",
                "Quality": "quality_id",
                "SubQuality": "subquality_id"}, 
                axis = 'columns')
                

## Merge codes into main df

# Country
tradeflow = pd.merge_ordered(tradeflow, country_id, left_on = ["country_id"], right_on = ["country_id"])

# Shuffle names for columns to get the destination country as well
tradeflow = tradeflow.rename({"country_id": "source_country_id", 
                              "Country": "SourceCountry",
                              "DestinationCountry": "country_id"},
                              axis = 'columns')
tradeflow = pd.merge_ordered(tradeflow, country_id, left_on = ["country_id"], right_on = ["country_id"])

# Commodity
tradeflow = pd.merge_ordered(tradeflow, commodity_id, left_on = ["commodity_id"], right_on = ["commodity_id"])
tradeflow = pd.merge_ordered(tradeflow, commodity_quality_id, left_on = ["quality_id", "commodity_id"], right_on = ["quality_id", "commodity_id"])
tradeflow = pd.merge_ordered(tradeflow, commodity_sub_quality_id, left_on = ["subquality_id", "quality_id"], right_on = ["subquality_id", "quality_id"])

# Rename to match original tradeflows name
tradeflow = tradeflow.rename({"CommodityQuality": "Quality", 
                              "CommoditySubQuality": "SubQuality",
                              "Country": "DestinationCountry",
                              "Status": "status_id"},
                              axis = 'columns')



#### Get rid of unmapped countries

tradeflow_wheat = tradeflow.loc[tradeflow["Commodity"] == "Wheat"]
tradeflow_wheat = tradeflow_wheat.loc[tradeflow_wheat["Quality"] == "All Non-durum"]
tradeflow_wheat = tradeflow_wheat.loc[tradeflow_wheat["SourceCountry"].isin(want_countries)]
# tradeflow_wheat = tradeflow_wheat.loc[tradeflow_wheat["SubQuality"] == "Wheat"]


# # Get rid of null entries
tradeflow_wheat = tradeflow_wheat.loc[tradeflow_wheat["country_id"].notna()]

# # Keep only countries on my list

#### Map status

# Rename status matrix
status_id = status_id.rename({"Id": "status_id",
                  "Name": "Status"}, 
                 axis = 1)

# Merge
tradeflow_wheat = pd.merge_ordered(tradeflow_wheat, status_id, left_on = ["status_id"], right_on = ["status_id"])

#### Cleanup


# Keep only shipped and at-price
# tradeflow_wheat = tradeflow_wheat.loc[tradeflow_wheat["Status"].isin(["Shipped", "Expected"])]


# Helper column for market year
tradeflow_wheat["my_start"] = tradeflow_wheat["SourceCountry"].map(cropyearstart_barley)
tradeflow_wheat["newcrop"] = tradeflow_wheat["Month"] >= tradeflow_wheat["my_start"]

# Get market year
tradeflow_wheat["my"] = tradeflow_wheat["Year"]
tradeflow_wheat["my"].loc[~tradeflow_wheat["newcrop"]] = tradeflow_wheat["my"] - 1



# Get rid of countries without a proper market year mapped
# tradeflow = tradeflow.loc[tradeflow["my_start"].notna()]


# Select columns
tradeflow_wheat = tradeflow_wheat[["SourceCountry",  "DestinationCountry", "Year", "Month", "Value", "my", "newcrop", "my_start", "Status", "IsExporter"]]

def assign_season_barley(row):
    year = row['Year']
    month = row['Month']


    if pd.isna(year) or pd.isna(month):
        return None  # or "Unknown"

    year = int(year)
    month = int(month)

    if month >= 12:
        season_start = year
    else:
        season_start = year - 1

    return f"{season_start}/{season_start + 1}"

# Apply to create the new column
tradeflow_wheat['Season'] = tradeflow_wheat.apply(assign_season_barley, axis=1)


tradeflow_wheat_current_season=tradeflow_wheat.loc[tradeflow_wheat['Season'] == current_season]
tradeflow_wheat_current_season=tradeflow_wheat_current_season.drop(columns=['SourceCountry','my','my_start','Season','newcrop','Year'])
tradeflow_wheat_current_season["DestinationCountry"] = tradeflow_wheat_current_season["DestinationCountry"].replace('ZZZ_Unknown_Destination', 'UNKNOWN')

# Select columns
tradeflow_wheat = tradeflow_wheat[["SourceCountry",  "DestinationCountry", "Year", "Month", "Value", "my", "newcrop", "my_start", "Status", "IsExporter"]]
tradeflow_wheat = tradeflow_wheat[tradeflow_wheat["my"].notna()]
tradeflow_wheat["my"] = tradeflow_wheat["my"].round().astype(int)
tradeflow_wheat["Season"] = tradeflow_wheat["my"].astype(str) + "/" + (tradeflow_wheat["my"] + 1).astype(str)



tradeflow_wheat_current_season=tradeflow_wheat.loc[tradeflow_wheat['Season'] == current_season]
tradeflow_wheat_current_season=tradeflow_wheat_current_season.drop(columns=['SourceCountry','my','my_start','Season','newcrop','Year'])
tradeflow_wheat_current_season["DestinationCountry"] = tradeflow_wheat_current_season["DestinationCountry"].replace('ZZZ_Unknown_Destination', 'UNKNOWN')

filtered_tradeflow_wheat_current_season = tradeflow_wheat_current_season[tradeflow_wheat_current_season['Status'].isin(['Expected', 'Shipped'])&(tradeflow_wheat_current_season['IsExporter'] == True)]

filtered_tradeflow_wheat_current_season.drop(columns=['Status','IsExporter'])


tradeflow_wheat_current_season_pivot = filtered_tradeflow_wheat_current_season.pivot_table(
    index='DestinationCountry',
    columns='Month',
    values='Value',
    aggfunc='sum'  # or 'mean', 'first', etc. depending on your need
)

tradeflow_wheat_current_season_pivot.columns.name = None
tradeflow_wheat_current_season_pivot.columns = tradeflow_wheat_current_season_pivot.columns.astype(int)
tradeflow_wheat_current_season_pivot



# Reorder columns: March to December, then January and February
ordered_cols = [12]+list(range(1, 12)) 
tradeflow_wheat_current_season_pivot = tradeflow_wheat_current_season_pivot[ordered_cols]

world_row = tradeflow_wheat_current_season_pivot.sum(numeric_only=True)

# Assign it to a new row named 'WORLD'
tradeflow_wheat_current_season_pivot.loc['WORLD'] = world_row



tradeflow_wheat_current_season_pivot.reset_index(inplace=True)
tradeflow_wheat_current_season_pivot['DestinationCountry'] = tradeflow_wheat_current_season_pivot['DestinationCountry'].apply(lambda x: country_mapping.get(x, x))
tradeflow_wheat_current_season_pivot.rename(columns={'DestinationCountry': '0'}, inplace=True)




In [0]:
pivot_df=tradeflow_wheat_current_season_pivot.copy()


In [0]:


# Rename first column to 'Country'
df1 = monthly_table.rename(columns={monthly_table.columns[0]: "Country"})
df2 = pivot_df.rename(columns={pivot_df.columns[0]: "Country"})

# Normalize country names to uppercase (or lowercase, your choice)
df1["Country"] = df1["Country"].str.upper()
df2["Country"] = df2["Country"].str.upper()

# Extract the month (as an integer)
month_col = selected_month


# Extract and rename relevant columns
lineups = df1[["Country", month_col]].rename(columns={month_col: "Lineup"})
forecasts = df2[["Country", month_col]].rename(columns={month_col: "Forecast"})

# Merge
merged = pd.merge(lineups, forecasts, on="Country", how="outer")

# Optional: sort
merged = merged.sort_values("Country").reset_index(drop=True)

merged['Forecast']=merged['Forecast']*1000
merged['Lineup']=merged['Lineup'].round()

merged['Country'] = merged['Country'].apply(lambda x: country_mapping.get(x, x))
merged['Region'] = merged['Country'].map(country_region_mapping)
merged['Var']=merged['Lineup']-merged['Forecast']

# 1. Identify and extract rows for KENYA and UGANDA
kenya_uganda_mask = merged['Country'].isin(['KENYA', 'UGANDA'])
kenya_uganda_rows = merged[kenya_uganda_mask]

# 2. Sum Lineup, Forecast, Var for KENYA and UGANDA
combined_row = kenya_uganda_rows[['Lineup', 'Forecast', 'Var']].sum()
combined_row['Country'] = 'KENYA+UGANDA'

# Use the first non-null region if both countries are in the same region (likely 'AFRICA')
combined_row['Region'] = kenya_uganda_rows['Region'].dropna().iloc[0]

# 3. Drop KENYA and UGANDA from the original dataframe
merged = merged[~kenya_uganda_mask]

# 4. Append the new combined row
merged = pd.concat([merged, pd.DataFrame([combined_row])], ignore_index=True)

# Group by 'Region' and calculate the sum for each region
region_subtotals = merged.groupby('Region').agg({
    'Lineup': 'sum',
    'Forecast': 'sum',
    'Var': 'sum'
}).reset_index()

# Add a column for the subtotal row name
region_subtotals['Country'] = 'Subtotal'

# Append the subtotal rows to the merged dataframe
merged_with_subtotals = pd.concat([merged, region_subtotals], ignore_index=True)

# Sort the dataframe to place subtotal rows at the end of each region
merged_with_subtotals['Region_Order'] = merged_with_subtotals['Region'].map({
    'AFRICA': 0, 'ASIA': 1, 'CENTRAL AMERICA': 2, 'EU': 3,
    'EUROPE NON-EU': 4,'FSU':5, 'MIDDLE EAST': 6, 'NORTH AMERICA': 7,'OCEANIA': 8,'SOUTH AMERICA': 9,'UNKNOWN':10,"WORLD":11
})

merged_with_subtotals = merged_with_subtotals.sort_values(by=['Region_Order', 'Country'])

# Drop the 'Region_Order' column
merged_with_subtotals = merged_with_subtotals.drop(columns=['Region_Order'])


region_order = ['AFRICA', 'ASIA', 'CENTRAL AMERICA', 'EU','EUROPE NON-EU','FSU', 'MIDDLE EAST', 'NORTH AMERICA','OCEANIA','SOUTH AMERICA','UNKNOWN',"WORLD"]

# Ensure Region is a categorical column with order
merged_with_subtotals['Region'] = pd.Categorical(
    merged_with_subtotals['Region'],
    categories=region_order,
    ordered=True
)

# Replace 'Subtotal' country entries with 'Subtotal [Region]'
merged_with_subtotals.loc[
    merged_with_subtotals['Country'] == 'Subtotal',
    'Country'
] = 'Subtotal ' + merged_with_subtotals['Region'].astype(str)

# Sort values by Region and then within Region put Subtotal at the end
def custom_sort(df):
    # Put all non-subtotals first, then the subtotal
    subtotals = df[df['Country'].str.startswith('Subtotal')]
    others = df[~df['Country'].str.startswith('Subtotal')]
    return pd.concat([others, subtotals])

# Apply the sorting logic by region
merged_with_subtotals = (
    merged_with_subtotals
    .sort_values(['Region', 'Country'])  # Preliminary sort
    .groupby('Region', group_keys=False)
    .apply(custom_sort)
)
merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal UNKNOWN']
merged_with_subtotals = merged_with_subtotals[merged_with_subtotals['Country'] != 'Subtotal WORLD']

# Optionaleset index or keep Region as index
merged_with_subtotals.set_index(['Region', 'Country'], inplace=True)

total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

# Set the value for the WORLD row
merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] = total_lineup


merged_with_subtotals = merged_with_subtotals.rename(columns={'Forecast': 'BS'})

# Now set the 'Var' column for the 'WORLD' row
merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Var'] = merged_with_subtotals.loc[('WORLD', 'WORLD'), 'Lineup'] - merged_with_subtotals.loc[('WORLD', 'WORLD'), 'BS']


merged_with_subtotals_ht=merged_with_subtotals.fillna(0)

# Drop rows where both Lineup and Forecast are zero
merged_with_subtotals_ht= merged_with_subtotals_ht[~((merged_with_subtotals_ht['Lineup'] == 0) & (merged_with_subtotals_ht['BS'] == 0))]

# Format numeric values with thousand separators and no decimals
merged_with_subtotals_ht[['Lineup', 'BS', 'Var']] = merged_with_subtotals_ht[['Lineup', 'BS', 'Var']].applymap(lambda x: f"{x/1000:,.0f}k")

merged_with_subtotals_ht.rename(columns={'BS':'At Price'},inplace=True)

total_lineup = merged_with_subtotals[~merged_with_subtotals.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()




#### NEXT MONTH

In [0]:
from dateutil.relativedelta import relativedelta

In [0]:
# pivot_df=tradeflow_current_season_pivot.copy()


# # Replace values in columns
# pivot_df["0"] = pivot_df["0"].replace('ZZZ_Unknown', 'UNKNOWN')
# pivot_df["0"] = pivot_df["0"].replace('World: total areas', 'WORLD')

# # Rename first column to 'Country'
# df1 = monthly_table.rename(columns={monthly_table.columns[0]: "Country"})
# df2 = pivot_df.rename(columns={pivot_df.columns[0]: "Country"})

# # Normalize country names to uppercase (or lowercase, your choice)
# df1["Country"] = df1["Country"].str.upper()
# df2["Country"] = df2["Country"].str.upper()

# # Select current month
# target_date = datetime.today() + relativedelta(months=1)

# # Extract the month (as an integer)
# month_col = target_date.month

# lineups = df1[["Country"]].copy()
# lineups["Lineup"] = df1[month_col] if month_col in df1.columns else 0

# forecasts = df2[["Country"]].copy()
# forecasts["Forecast"] = df2[month_col] if month_col in df2.columns else 0

# # Merge
# merged_next = pd.merge(lineups, forecasts, on="Country", how="outer")

# # Optional: sort
# merged_next = merged_next.sort_values("Country").reset_index(drop=True)

# merged_next['Forecast']=merged_next['Forecast']*1000
# merged_next['Lineup']=merged_next['Lineup'].round()
# merged_next['Var']=merged_next['Lineup']-merged_next['Forecast']

# merged_next['Country'] = merged_next['Country'].apply(lambda x: country_mapping.get(x, x))
# merged_next['Region'] = merged_next['Country'].map(country_region_mapping)


# # Group by 'Region' and calculate the sum for each region
# region_subtotals_next = merged_next.groupby('Region').agg({
#     'Lineup': 'sum',
#     'Forecast': 'sum',
#     'Var': 'sum'
# }).reset_index()

# # Add a column for the subtotal row name
# region_subtotals_next['Country'] = 'Subtotal'

# # Append the subtotal rows to the merged_next dataframe
# merged_with_subtotals_next = pd.concat([merged_next, region_subtotals_next], ignore_index=True)

# # Sort the dataframe to place subtotal rows at the end of each region
# merged_with_subtotals_next['Region_Order'] = merged_with_subtotals_next['Region'].map({
#     'NORTH AMERICA': 0, 'CENTRAL AMERICA': 1, 'SOUTH AMERICA': 2, 'EU': 3,
#     'ME ASIA': 4, 'AFRICA': 5, 'ASIA': 6, 'SE ASIA': 7, 'OCEANIA': 8,'UNKNOWN':9,"WORLD":10
# })

# merged_with_subtotals_next = merged_with_subtotals_next.sort_values(by=['Region_Order', 'Country'])

# # Drop the 'Region_Order' column
# merged_with_subtotals_next = merged_with_subtotals_next.drop(columns=['Region_Order'])

# region_order = [
#     'NORTH AMERICA', 'CENTRAL AMERICA', 'SOUTH AMERICA',
#     'EU', 'ME ASIA', 'AFRICA', 'ASIA', 'SE ASIA', 'OCEANIA','UNKNOWN',"WORLD"
# ]

# # Ensure Region is a categorical column with order
# merged_with_subtotals_next['Region'] = pd.Categorical(
#     merged_with_subtotals_next['Region'],
#     categories=region_order,
#     ordered=True
# )

# # Replace 'Subtotal' country entries with 'Subtotal [Region]'
# merged_with_subtotals_next.loc[
#     merged_with_subtotals_next['Country'] == 'Subtotal',
#     'Country'
# ] = 'Subtotal ' + merged_with_subtotals_next['Region'].astype(str)

# # Sort values by Region and then within Region put Subtotal at the end
# def custom_sort(df):
#     # Put all non-subtotals first, then the subtotal
#     subtotals = df[df['Country'].str.startswith('Subtotal')]
#     others = df[~df['Country'].str.startswith('Subtotal')]
#     return pd.concat([others, subtotals])

# # Apply the sorting logic by region
# merged_with_subtotals_next = (
#     merged_with_subtotals_next
#     .sort_values(['Region', 'Country'])  # Preliminary sort
#     .groupby('Region', group_keys=False)
#     .apply(custom_sort)
# )
# merged_with_subtotals_next = merged_with_subtotals_next[merged_with_subtotals_next['Country'] != 'Subtotal UNKNOWN']
# merged_with_subtotals_next = merged_with_subtotals_next[merged_with_subtotals_next['Country'] != 'Subtotal WORLD']

# # Optional: reset index or keep Region as index
# merged_with_subtotals_next.set_index(['Region', 'Country'], inplace=True)

# total_lineup = merged_with_subtotals_next[~merged_with_subtotals_next.index.get_level_values('Country').str.contains('Subtotal')]['Lineup'].sum()

# # Set the value for the WORLD row
# merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'Lineup'] = total_lineup


# merged_with_subtotals_next = merged_with_subtotals_next.rename(columns={'Forecast': 'BS'})

# # Now set the 'Var' column for the 'WORLD' row
# merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'Var'] = merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'Lineup'] - merged_with_subtotals_next.loc[('WORLD', 'WORLD'), 'BS']


# merged_with_subtotals_ht_next=merged_with_subtotals_next.fillna(0)

# # Drop rows where both Lineup and Forecast are zero
# merged_with_subtotals_ht_next= merged_with_subtotals_ht_next[~((merged_with_subtotals_ht_next['Lineup'] == 0) & (merged_with_subtotals_ht_next['BS'] == 0))]

# # Format numeric values with thousand separators and no decimals
# merged_with_subtotals_ht_next[['Lineup', 'BS', 'Var']] = merged_with_subtotals_ht_next[['Lineup', 'BS', 'Var']].applymap(lambda x: f"{x/1000:,.0f}k")

# merged_with_subtotals_ht_next.rename(columns={'BS':'At Price'},inplace=True)



In [0]:

merged_with_subtotals_html=merged_with_subtotals_ht.to_html()



# merged_with_subtotals_html_next=merged_with_subtotals_ht_next.to_html()

### CHART OVERVIEW

In [0]:
from calendar import monthrange

BS_world = merged_with_subtotals.reset_index()
BS_value = BS_world.loc[BS_world['Country'] == 'WORLD', 'BS'].values[0]

filtered_df_W = filtered_df_W[filtered_df_W['Origin']=='ARG']

sailed_df = filtered_df_W[filtered_df_W['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()

# Step 1: Filter for Status being 'SAILED' or 'ANNOUNCED'
forecast_df = filtered_df_W[filtered_df_W['Status'].isin(['SAILED', 'ANNOUNCED'])].copy()

# Step 2: Make sure 'Date' is datetime
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])

# Step 3: Sort by date
forecast_df = forecast_df.sort_values(by='Date')

# Step 4: Compute cumulative sum of Quantity
forecast_df['Forecasts'] = forecast_df['Quantity'].cumsum()

daily_totals_sailed = sailed_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_sailed['Sailed'] = daily_totals_sailed['Quantity'].cumsum()

last_sailed=daily_totals_sailed.iloc[-1]
daily_totals_forecast = forecast_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()
daily_totals_forecast['Date'] = pd.to_datetime(daily_totals_forecast['Date']).dt.date

# Extract base year from season string
base_year = int(selected_season.split("/")[0])

# Determine actual year for the month
if selected_month in [12]:
    year = base_year
else:
    year = base_year +1

# Get number of days in that month
num_days = monthrange(year, selected_month)[1]

# Create date range
start_date = datetime(year, selected_month, 1)
end_date = datetime(year, selected_month, num_days)
all_dates = pd.date_range(start=start_date, end=end_date, freq='D').date



# Ensure forecast date has no time
daily_totals_forecast['Date'] = pd.to_datetime(daily_totals_forecast['Date']).dt.date

# Create DataFrame of full month dates
full_dates_df = pd.DataFrame({'Date': all_dates})

# Merge to preserve values and fill missing dates
daily_totals_forecast = full_dates_df.merge(
    daily_totals_forecast, on='Date', how='left'
)

# Fill missing Quantity values
daily_totals_forecast['Quantity'] = daily_totals_forecast['Quantity'].fillna(0)

# Recalculate cumulative Forecasts
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()


daily_totals_forecast['BS_per_day'] = BS_value / num_days
daily_totals_forecast['BS'] = daily_totals_forecast['BS_per_day'].cumsum()

last_forecast = daily_totals_forecast.iloc[-1]

In [0]:
from calendar import monthrange

BS_world = merged_with_subtotals.reset_index()
BS_value = BS_world.loc[BS_world['Country'] == 'WORLD', 'BS'].values[0]

filtered_df_W = filtered_df_W[filtered_df_W['Origin']=='ARG']

sailed_df = filtered_df_W[filtered_df_W['Status'] == 'SAILED'].copy()

# Step 2: Make sure 'Date' is a datetime column
sailed_df['Date'] = pd.to_datetime(sailed_df['Date'])

# Step 3: Sort by 'Date'
sailed_df = sailed_df.sort_values(by='Date')

# Step 4: Compute the cumulative sum of 'Quantity'
sailed_df['Sailed'] = sailed_df['Quantity'].cumsum()

# Step 1: Filter for Status being 'SAILED' or 'ANNOUNCED'
forecast_df = filtered_df_W[filtered_df_W['Status'].isin(['SAILED', 'ANNOUNCED'])].copy()

# Step 2: Make sure 'Date' is datetime
forecast_df['Date'] = pd.to_datetime(forecast_df['Date'])

# Step 3: Sort by date
forecast_df = forecast_df.sort_values(by='Date')

# Step 4: Compute cumulative sum of Quantity
forecast_df['Forecasts'] = forecast_df['Quantity'].cumsum()

daily_totals_sailed = sailed_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_sailed['Sailed'] = daily_totals_sailed['Quantity'].cumsum()

if len(daily_totals_sailed) < 15:
    # Get year and month from the current Date
    only_date = daily_totals_sailed.loc[0, 'Date']
    year = only_date.year
    month = only_date.month
    
    # Get last day of that month
    last_day = monthrange(year, month)[1]
    last_date = datetime(year, month, last_day)
    idx_max = daily_totals_sailed['Quantity'].idxmax()
    daily_totals_sailed = daily_totals_sailed.loc[[idx_max]].copy()

    # Reassign
    daily_totals_sailed.loc[0, 'Date'] = last_date

    
last_sailed= daily_totals_sailed.iloc[-1]

daily_totals_forecast = forecast_df.groupby('Date', as_index=False)['Quantity'].sum()

# Step 2: Compute the cumulative sum
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()
daily_totals_forecast['Date'] = pd.to_datetime(daily_totals_forecast['Date']).dt.date

# Extract base year from season string
base_year = int(selected_season.split("/")[0])

# Determine actual year for the month
if selected_month in [12]:
    year = base_year 
else:
    year = base_year+1

# Get number of days in that month
num_days = monthrange(year, selected_month)[1]

# Create date range
start_date = datetime(year, selected_month, 1)
end_date = datetime(year, selected_month, num_days)
all_dates = pd.date_range(start=start_date, end=end_date, freq='D').date

# Ensure forecast date has no time
daily_totals_forecast['Date'] = pd.to_datetime(daily_totals_forecast['Date']).dt.date

# Create DataFrame of full month dates
full_dates_df = pd.DataFrame({'Date': all_dates})

# Merge to preserve values and fill missing dates
daily_totals_forecast = full_dates_df.merge(
    daily_totals_forecast, on='Date', how='left'
)

# Fill missing Quantity values
daily_totals_forecast['Quantity'] = daily_totals_forecast['Quantity'].fillna(0)

# Recalculate cumulative Forecasts
daily_totals_forecast['Forecasts'] = daily_totals_forecast['Quantity'].cumsum()


daily_totals_forecast['BS_per_day'] = BS_value / num_days
daily_totals_forecast['BS'] = daily_totals_forecast['BS_per_day'].cumsum()


In [0]:

overview = go.Figure()

# Sailed line
overview.add_trace(go.Scatter(
    x=daily_totals_sailed['Date'],
    y=daily_totals_sailed['Sailed'],
    mode='lines+markers',
    name='Sailed',
    line=dict(color='blue')
))



# Forecast line
overview.add_trace(go.Scatter(
    x=daily_totals_forecast['Date'],
    y=daily_totals_forecast['BS'],
    mode='lines+markers',
    name='Export Forecasts',
    line=dict(color='red', dash='dashdot')
))


# Add annotations
overview.add_annotation(
    x=last_sailed['Date'],
    y=last_sailed['Sailed'],
    text=f"{int(last_sailed['Sailed']/1000):,} kmt",
    showarrow=True,
    arrowhead=1,
    ax=0,
    ay=-60,
    font=dict(color="blue")
)


overview.add_annotation(
    x=last_forecast['Date'],
    y=last_forecast['BS'],
    text=f"{int(last_forecast['BS']/1000):,} kmt",
    showarrow=True,
    arrowhead=1,
    ax=70,
    ay=-20,
    font=dict(color="red")
)


# Layout
overview.update_layout(
    title='Wheat exports Sailed vs Forecasts',
    xaxis_title='Date',
    yaxis_title='Quantity',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99),
    height=700,
    width=1200,
    hovermode='x unified'
)


overview.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')


# Export to image
pio.write_image(overview, "sailed_vs_forecast.png", width=1000, height=600)

overview.show()


overview_html = overview.to_html(include_plotlyjs='cdn', full_html=True)

### SUMMARY DF

In [0]:
filtered_df_W
sailed_sum = filtered_df_W[filtered_df_W['Status'] == 'SAILED']['Quantity'].sum()
announced_sum = filtered_df_W[filtered_df_W['Status'] == 'ANNOUNCED']['Quantity'].sum()
total=sailed_sum+announced_sum 
forecast_BS=last_forecast['BS']
percent_vs_mars=sailed_sum/forecast_BS*100

summary_df = pd.DataFrame([{
    'Sailed': f"{round(sailed_sum / 1000)} kmt",
    'Announced': f"{round(announced_sum / 1000)} kmt",
    'Total': f"{round(total / 1000)} kmt",
    'MARS': f"{round(forecast_BS / 1000) }kmt",
    '% vs MARS': round(percent_vs_mars)
}])


In [0]:
summary_df_html=summary_df.to_html(classes='table table-striped table-bordered', index=False)

In [0]:
# Ensure month column exists, otherwise fill with zeros
anncd = monthly_table_ancd[["Destination"]].copy()
anncd["Announced"] = monthly_table_ancd.get(month_col, pd.Series([0]*len(monthly_table_ancd)))

sailed = monthly_table_sailed[["Destination"]].copy()
sailed["Sailed"] = monthly_table_sailed.get(month_col, pd.Series([0]*len(monthly_table_sailed)))

anncd.rename(columns={'Destination': 'Country'}, inplace=True)
sailed.rename(columns={'Destination': 'Country'}, inplace=True)

anncd['Country'] = anncd['Country'].apply(lambda x: country_mapping.get(x, x))
sailed['Country'] = sailed['Country'].apply(lambda x: country_mapping.get(x, x))

# 1. Identify and extract rows for KENYA and UGANDA
kenya_uganda_mask = sailed['Country'].isin(['KENYA', 'UGANDA'])
kenya_uganda_rows = sailed[kenya_uganda_mask]

# 2. Sum Lineup, Forecast, Var for KENYA and UGANDA
combined_row = kenya_uganda_rows[['Sailed']].sum()
combined_row['Country'] = 'KENYA+UGANDA'

# 3. Drop KENYA and UGANDA from the original dataframe
sailed = sailed[~kenya_uganda_mask]

# 4. Append the new combined row
sailed = pd.concat([sailed, pd.DataFrame([combined_row])], ignore_index=True)

merged_bar=pd.merge(anncd, sailed, on='Country',how='outer')

merged_bar=pd.merge(merged, merged_bar, on='Country', how='outer')
merged_bar=merged_bar.sort_values('Region')

excluded_destinations = ['SOUTH AMERICA', 'UNKNOWN', 'WORLD', 'AFRICA', 'ZZZ_UNKNOWN_DESTINATION']
merged_bar = merged_bar[~merged_bar['Country'].isin(excluded_destinations)]

# Filter out rows where Sailed, Announced, and Forecast are all 0 or NaN
merged_bar = merged_bar[~(((merged_bar['Sailed'].fillna(0) == 0) & (merged_bar['Announced'].fillna(0) == 0) & (merged_bar['Forecast'].fillna(0) == 0)))]

merged_bar['Total'] = merged_bar['Sailed'].fillna(0) + merged_bar['Announced'].fillna(0)

# Sort by this total and keep only top 20
merged_bar = merged_bar.sort_values('Total', ascending=False).head(20)

# Optional: sort again for cleaner plotting (descending)
merged_bar = merged_bar.sort_values('Total', ascending=True) 

# Step 1: Create the base chart with Sailed + Announced side by side
bar = go.Figure()

# Sailed
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Sailed'],
    orientation='h',
    name='Sailed',
    marker=dict(color='steelblue'),
    offsetgroup=0,
    base=0,
    text=merged_bar['Sailed'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=30)  
))


# Step 2: Add Forecast with overlay
bar.add_trace(go.Bar(
    y=merged_bar['Country'],
    x=merged_bar['Forecast'],
    orientation='h',
    name='Forecasts',
    marker=dict(color='rgba(255, 0, 0, 0.4)'),  # Transparent red
    offsetgroup=1,
    base=0,
    opacity=0.4,
    text=merged_bar['Forecast'].apply(lambda x: f"{x:,.0f}"),  # Adding text to bars
    textposition='inside',  # Position text inside the bar
    textfont=dict(size=60),
))

# Layout
bar.update_layout(
    barmode='group',  # This allows Forecast to overlap
    title='Sailed et Announced vs Balance Sheet by Country',
    xaxis_title='Quantity (mt)',
    yaxis_title='Country',
    template='plotly_white',
    height=900,
    width=900,
)

bar.show()

bar_html = bar.to_html(include_plotlyjs='cdn', full_html=True)

### TOP 10

In [0]:
# Add new column for the total of other columns
monthly_table['Total'] = monthly_table.iloc[:, 1:].sum(axis=1)

# Step 2: Select top 10 countries with the highest total
top_10_countries = monthly_table.nlargest(10, 'Total')

marketing_month_order = [12,1,2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
marketing_month_names = ['Dec', 'Jan', 'Feb','Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov']
# Step 4: Determine which month columns are currently in the DataFrame (as numbers)
month_cols = [col for col in marketing_month_order if col in monthly_table.columns]

# Step 5: Create the new column labels (starting with 'Destination' and ending with 'Total')
new_columns = ['Destination'] + [marketing_month_names[marketing_month_order.index(m)] for m in month_cols] + ['Total']

# Step 6: Apply new column names to top_10_countries
top_10_countries.columns = new_columns

# Add Total column
monthly_table['Total'] = monthly_table.iloc[:, 1:].sum(axis=1)

# Select top 10 countries
top_10_countries = monthly_table.nlargest(10, 'Total')

month_cols = [m for m in marketing_month_order if m in monthly_table.columns]
top_10_countries = top_10_countries[['Destination'] + month_cols + ['Total']]
top_10_countries.columns = ['Destination'] + [marketing_month_names[marketing_month_order.index(m)] for m in month_cols] + ['Total']


In [0]:
import plotly.graph_objects as go

# Step 1: Determine month labels based on columns (excluding 'Destination' and 'Total')
month_labels = new_columns[1:-1]  # Already ordered and renamed in previous steps

# Step 2: Create the plot
top = go.Figure()

# Step 3: Add a trace for each of the top 10 countries
for idx, row in top_10_countries.iterrows():
    top.add_trace(go.Scatter(
        x=month_labels,
        y=row[1:-1],  # skip 'Destination' and 'Total'
        mode='lines+markers',
        name=row['Destination']
    ))

# Step 4: Update the layout
top.update_layout(
    title='Top 10 Barley Destinations by Month',
    xaxis_title='Month',
    yaxis_title='Quantity (mt)',
    template='plotly_white',
    height=600,
    width=950,
    hovermode='x unified'
)

top.show()
top_html = top.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
# Function to format numbers
def format_kmt(value):
    try:
        num = float(str(value).replace(',', ''))
        return f"{num / 1000:.1f}k"
    except:
        return value  # in case it's non-numeric


# Apply formatting to all numeric columns
for col in top_10_countries.columns[1:]:  # Skip 'Destination'
    top_10_countries[col] = top_10_countries[col].apply(format_kmt)

top_10_countries.reset_index(inplace=True)
top_10_countries = top_10_countries.drop(columns='index')
top_10_countries_html=top_10_countries.to_html()


### SHIPPERS AND COORDINATORS FIGURES OVER THE YEAR

In [0]:
year

In [0]:
current_year=year

# Filter for current season
df_current_year = history[history['Year'] == current_year]

# Extract marketing month (Dec to Nov)
df_current_year['Marketing_Month'] = df_current_year['Date'].dt.month

# Map month numbers to names, Dec to Nov
month_order = [1,2,3, 4, 5, 6, 7, 8, 9, 10, 11,12]
month_names = ['Jan', 'Feb','Mar', 'Apr', 'May', 'Jun','Jul', 'Aug', 'Sep', 'Oct', 'Nov','Dec']
month_map = dict(zip(month_order, month_names))

df_current_year['Rank'] = df_current_year['Marketing_Month'].map(month_map)

# Replace missing Coordinators with 'UNKNOWN' if needed
df_current_year['Coordinator'] = df_current_year.get('Coordinator', pd.Series()).fillna('UNKNOWN')

# Group by Coordinator and Rank and sum Quantity
pivot_df_current_year_coord = df_current_year.groupby(['Coordinator', 'Rank'])['Quantity'].sum().reset_index()

# Pivot to get desired format
pivot_table_coord = pivot_df_current_year_coord.pivot(index='Coordinator', columns='Rank', values='Quantity')

# Reorder columns to match Dec–Nov sequence
pivot_table_coord = pivot_table_coord.reindex(columns=month_names, fill_value=0)

# Rename index
pivot_table_coord.index.name = 'Coordinators'

# Optional: reset index if you want Coordinators as a column
pivot_table_coord = pivot_table_coord.reset_index()


# Add Total column (row-wise sum)
pivot_table_coord['Total'] = pivot_table_coord[month_names].sum(axis=1)

# Sort by Total descending and keep only top 20
top_20_coord = pivot_table_coord.sort_values(by='Total', ascending=False).head(20)

top_20_coord.insert(1, 'Rank', range(1, len(top_20_coord) + 1))

top_20_coord_html=top_20_coord.to_html()


%md
### NEUTRLIZED BELOW TO GET JAN-DEC OVERVIEW INSTEAD OF SEASON

In [0]:
# df_current_season = history[history['Season'] == current_season]

# # Extract marketing month (Dec to Nov)
# df_current_season['Marketing_Month'] = df_current_season['Date'].dt.month

# # Map month numbers to names, Dec to Nov
# month_order = [3, 4, 5, 6, 7, 8, 9, 10, 11,12,1,2]
# month_names = ['Mar', 'Apr', 'May', 'Jun','Jul', 'Aug', 'Sep', 'Oct', 'Nov','Dec', 'Jan', 'Feb',]
# month_map = dict(zip(month_order, month_names))

# df_current_season['Rank'] = df_current_season['Marketing_Month'].map(month_map)

# # Replace missing Coordinators with 'UNKNOWN' if needed
# df_current_season['Coordinator'] = df_current_season.get('Coordinator', pd.Series()).fillna('UNKNOWN')

# # Group by Coordinator and Rank and sum Quantity
# pivot_df_current_season_coord = df_current_season.groupby(['Coordinator', 'Rank'])['Quantity'].sum().reset_index()

# # Pivot to get desired format
# pivot_table_coord = pivot_df_current_season_coord.pivot(index='Coordinator', columns='Rank', values='Quantity')

# # Reorder columns to match Dec–Nov sequence
# pivot_table_coord = pivot_table_coord.reindex(columns=month_names, fill_value=0)

# # Rename index
# pivot_table_coord.index.name = 'Coordinators'

# # Optional: reset index if you want Coordinators as a column
# pivot_table_coord = pivot_table_coord.reset_index()


# # Add Total column (row-wise sum)
# pivot_table_coord['Total'] = pivot_table_coord[month_names].sum(axis=1)

# # Sort by Total descending and keep only top 20
# top_20_coord = pivot_table_coord.sort_values(by='Total', ascending=False).head(20)

# top_20_coord.insert(1, 'Rank', range(1, len(top_20_coord) + 1))

# top_20_coord_html=top_20_coord.to_html()

In [0]:
coord_percent_df = top_20_coord.copy()
month_only = month_names  

# 2. Convertir les colonnes en valeurs numériques si elles sont formatées (optionnel)
# Si nécessaire, décommenter :
# for col in month_only:
#     coord_percent_df[col] = coord_percent_df[col].replace({',': ''}, regex=True).astype(float)

# 3. Calculer les totaux par mois (colonne)
monthly_totals = coord_percent_df[month_only].replace(',', '', regex=True).astype(float).sum(axis=0)

# 4. Calculer les pourcentages
percentage_table_coord = coord_percent_df.copy()
for col in month_only:
    percentage_table_coord[col] = coord_percent_df[col].replace(',', '', regex=True).astype(float) / monthly_totals[col] * 100

# 5. Formater les pourcentages
percentage_table_coord[month_only] = percentage_table_coord[month_only].applymap(lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%")

# 6. Compute and format % for the 'Total' column
total_volume = coord_percent_df['Total'].replace(',', '', regex=True).astype(float).sum()

percentage_table_coord['Total'] = coord_percent_df['Total'].replace(',', '', regex=True).astype(float) / total_volume * 100
percentage_table_coord['Total'] = percentage_table_coord['Total'].apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%")


percentage_table_coord_html = percentage_table_coord.to_html(index=False)

# Repartir des deux tables : `top_20_shipper` (quantités formatées) et `percentage_table` (pourcentages formatés)

# On crée une copie pour éviter toute modification accidentelle
combined_table_coord = top_20_coord.copy()

month_only = month_names+ ['Total']  # ['Dec', 'Jan', ..., 'Nov']

def format_kmt(value):
    try:
        if pd.isna(value) or str(value).strip() == '':
            return "0.0k"
        num = float(str(value).replace(',', ''))
        return f"{num / 1000:.1f}k"
    except:
        return "0.0k"

# Appliquer le format kmt aux colonnes mensuelles et combiner avec les pourcentages
# Appliquer le format kmt et combiner avec les pourcentages
for col in month_only:
    formatted_qty = top_20_coord[col].apply(format_kmt)
    formatted_pct = percentage_table_coord[col].fillna("0.0%")
    combined_table_coord[col] = formatted_qty + '  ' + formatted_pct


for col in month_only:
    combined_table_coord[col] = combined_table_coord[col].replace(['0.0k  0.0%', 'nank  0.0%', '0.0k (nan%)'], '')


combined_table_coord.reset_index(inplace=False)

combined_table_coord_html=combined_table_coord.to_html(index=False)


In [0]:
# Filter for current year
df_current_year = history[history['Year'] == current_year]
df_current_year = df_current_year[df_current_year['Product'] == 'Wheat']
# Extract marketing month
df_current_year['Marketing_Month'] = df_current_year['Date'].dt.month

# Month order & names
month_order = [1,2,3,4,5,6,7,8,9,10,11,12]
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_map = dict(zip(month_order, month_names))

df_current_year['Rank'] = df_current_year['Marketing_Month'].map(month_map)

# Replace missing Coordinators
df_current_year['Coordinator'] = df_current_year['Coordinator'].fillna('UNKNOWN')

# Group and pivot
pivot_df_current_year_coord = (
    df_current_year.groupby(['Coordinator', 'Rank'])['Quantity'].sum().reset_index()
)

pivot_table_coord = pivot_df_current_year_coord.pivot(
    index='Coordinator', columns='Rank', values='Quantity'
).reindex(columns=month_names, fill_value=0)

pivot_table_coord.index.name = 'Coordinators'
pivot_table_coord = pivot_table_coord.reset_index()

# Add totals
pivot_table_coord['Total'] = pivot_table_coord[month_names].sum(axis=1)

# === TOP 20 by total ===
top_20_coord = (
    pivot_table_coord.sort_values(by='Total', ascending=False).head(20).copy()
)
top_20_coord.insert(1, 'Rank', range(1, len(top_20_coord) + 1))

# === PERCENTAGES (relative to whole dataset) ===
monthly_totals = pivot_table_coord[month_names].sum(axis=0)
total_volume = pivot_table_coord['Total'].sum()

percentage_table_coord = top_20_coord.copy()
for col in month_names:
    percentage_table_coord[col] = (
        top_20_coord[col] / monthly_totals[col] * 100
    )

percentage_table_coord['Total'] = (
    top_20_coord['Total'] / total_volume * 100
)

# Format percentages
percentage_table_coord[month_names + ['Total']] = percentage_table_coord[month_names + ['Total']].applymap(
    lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%"
)

# === COMBINE FORMATTED QUANTITIES + PERCENTAGES ===
def format_kmt(value):
    try:
        if pd.isna(value) or str(value).strip() == '':
            return "0.0k"
        num = float(str(value).replace(',', ''))
        return f"{num/1000:.1f}k"
    except:
        return "0.0k"

combined_table_coord = top_20_coord.copy()
month_only = month_names + ['Total']

for col in month_only:
    formatted_qty = top_20_coord[col].apply(format_kmt)
    formatted_pct = percentage_table_coord[col].fillna("0.0%")
    combined_table_coord[col] = formatted_qty + '  ' + formatted_pct

# Clean up zeros
for col in month_only:
    combined_table_coord[col] = combined_table_coord[col].replace(
        ['0.0k  0.0%', 'nank  0.0%', '0.0k (nan%)'], ''
    )

# === FINAL HTML TABLE ===
combined_table_coord_html = combined_table_coord.to_html(index=False)


### NEUTRLIZED BELOW TO GET JAN-DEC OVERVIEW INSTEAD OF SEASON

In [0]:
# # Extract marketing month (Dec to Nov)
# df_current_season['Marketing_Month'] = df_current_season['Date'].dt.month

# df_current_season[' '] = df_current_season['Marketing_Month'].map(month_map)

# # Replace missing Coordinators with 'UNKNOWN' if needed
# df_current_season['Shipper'] = df_current_season.get('Shipper', pd.Series()).fillna('UNKNOWN')

# # Group by Shipper and   and sum Quantity
# pivot_df_current_season_shipper = df_current_season.groupby(['Shipper', ' '])['Quantity'].sum().reset_index()

# # Pivot to get desired format
# pivot_table_shipper = pivot_df_current_season_shipper.pivot(index='Shipper', columns=' ', values='Quantity')

# # Reorder columns to match Dec–Nov sequence
# pivot_table_shipper = pivot_table_shipper.reindex(columns=month_names, fill_value=0)

# # Rename index
# pivot_table_shipper.index.name = 'Shippers'

# # Optional: reset index if you want Coordinators as a column
# pivot_table_shipper = pivot_table_shipper.reset_index()


# # Add Total column (row-wise sum)
# pivot_table_shipper['Total'] = pivot_table_shipper[month_names].sum(axis=1)

# # Sort by Total descending and keep only top 20
# top_20_shipper = pivot_table_shipper.sort_values(by='Total', ascending=False).head(20)
# top_20_shipper.reset_index(inplace=True)
# top_20_shipper.drop(columns=['index'],inplace=True)


# top_20_shipper.iloc[:, 1:] = top_20_shipper.iloc[:, 1:].applymap(lambda x: f"{x:,.0f}")
# top_20_shipper_html=top_20_shipper.to_html()


In [0]:
# Extract marketing month (Dec to Nov)
df_current_year['Marketing_Month'] = df_current_year['Date'].dt.month

df_current_year[' '] = df_current_year['Marketing_Month'].map(month_map)

# Replace missing Coordinators with 'UNKNOWN' if needed
df_current_year['Shipper'] = df_current_year.get('Shipper', pd.Series()).fillna('UNKNOWN')

# Group by Shipper and   and sum Quantity
pivot_df_current_year_shipper = df_current_year.groupby(['Shipper', ' '])['Quantity'].sum().reset_index()

# Pivot to get desired format
pivot_table_shipper = pivot_df_current_year_shipper.pivot(index='Shipper', columns=' ', values='Quantity')

# Reorder columns to match Dec–Nov sequence
pivot_table_shipper = pivot_table_shipper.reindex(columns=month_names, fill_value=0)

# Rename index
pivot_table_shipper.index.name = 'Shippers'

# Optional: reset index if you want Coordinators as a column
pivot_table_shipper = pivot_table_shipper.reset_index()


# Add Total column (row-wise sum)
pivot_table_shipper['Total'] = pivot_table_shipper[month_names].sum(axis=1)

# Sort by Total descending and keep only top 20
top_20_shipper = pivot_table_shipper.sort_values(by='Total', ascending=False).head(20)
top_20_shipper.reset_index(inplace=True)
top_20_shipper.drop(columns=['index'],inplace=True)


top_20_shipper.iloc[:, 1:] = top_20_shipper.iloc[:, 1:].applymap(lambda x: f"{x:,.0f}")
top_20_shipper_html=top_20_shipper.to_html()


In [0]:
shipper_percent_df = top_20_shipper.copy()
month_only = month_names  # ['Dec', 'Jan', ..., 'Nov']

# 2. Convertir les colonnes en valeurs numériques si elles sont formatées (optionnel)
# Si nécessaire, décommenter :
# for col in month_only:
#     shipper_percent_df[col] = shipper_percent_df[col].replace({',': ''}, regex=True).astype(float)

# 3. Calculer les totaux par mois (colonne)
monthly_totals = shipper_percent_df[month_only].replace(',', '', regex=True).astype(float).sum(axis=0)

# 4. Calculer les pourcentages
percentage_table = shipper_percent_df.copy()
for col in month_only:
    percentage_table[col] = shipper_percent_df[col].replace(',', '', regex=True).astype(float) / monthly_totals[col] * 100

# 5. Formater les pourcentages
percentage_table[month_only] = percentage_table[month_only].applymap(lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%")

# 6. Compute and format % for the 'Total' column
total_volume = shipper_percent_df['Total'].replace(',', '', regex=True).astype(float).sum()

percentage_table['Total'] = shipper_percent_df['Total'].replace(',', '', regex=True).astype(float) / total_volume * 100
percentage_table['Total'] = percentage_table['Total'].apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%")


percentage_table_html = percentage_table.to_html(index=False)

In [0]:
# Repartir des deux tables : `top_20_shipper` (quantités formatées) et `percentage_table` (pourcentages formatés)

# On crée une copie pour éviter toute modification accidentelle
combined_table = top_20_shipper.copy()

month_only = month_names+ ['Total']  # ['Dec', 'Jan', ..., 'Nov']

def format_kmt(value):
    try:
        if pd.isna(value) or str(value).strip() == '':
            return "0.0k"
        num = float(str(value).replace(',', ''))
        return f"{num / 1000:.1f}k"
    except:
        return "0.0k"

# Appliquer le format kmt aux colonnes mensuelles et combiner avec les pourcentages
# Appliquer le format kmt et combiner avec les pourcentages
for col in month_only:
    formatted_qty = top_20_shipper[col].apply(format_kmt)
    formatted_pct = percentage_table[col].fillna("0.0%")
    combined_table[col] = formatted_qty + '  ' + formatted_pct


for col in month_only:
    combined_table[col] = combined_table[col].replace(['0.0k  0.0%', 'nank  0.0%', '0.0k (nan%)'], '')


combined_table.reset_index(inplace=False)

combined_table_html=combined_table.to_html(index=False)


In [0]:
# Replace missing Coordinators
df_current_year['Shipper'] = df_current_year['Shipper'].fillna('UNKNOWN')

# Group and pivot
pivot_df_current_year_shipper = (
    df_current_year.groupby(['Shipper', 'Rank'])['Quantity'].sum().reset_index()
)

pivot_table_shipper = pivot_df_current_year_shipper.pivot(
    index='Shipper', columns='Rank', values='Quantity'
).reindex(columns=month_names, fill_value=0)

pivot_table_shipper.index.name = 'Shippers'
pivot_table_shipper = pivot_table_shipper.reset_index()

# Add totals
pivot_table_shipper['Total'] = pivot_table_shipper[month_names].sum(axis=1)

# === TOP 20 by total ===
top_20_shipper = (
    pivot_table_shipper.sort_values(by='Total', ascending=False).head(20).copy()
)
top_20_shipper.insert(1, 'Rank', range(1, len(top_20_shipper) + 1))

# === PERCENTAGES (relative to whole dataset) ===
monthly_totals = pivot_table_shipper[month_names].sum(axis=0)
total_volume = pivot_table_shipper['Total'].sum()

percentage_table_shipper = top_20_shipper.copy()
for col in month_names:
    percentage_table_shipper[col] = (
        top_20_shipper[col] / monthly_totals[col] * 100
    )

percentage_table_shipper['Total'] = (
    top_20_shipper['Total'] / total_volume * 100
)

# Format percentages
percentage_table_shipper[month_names + ['Total']] = percentage_table_shipper[month_names + ['Total']].applymap(
    lambda x: f"{x:.1f}%" if pd.notnull(x) else "0.0%"
)

combined_table_shipper = top_20_shipper.copy()
month_only = month_names + ['Total']

for col in month_only:
    formatted_qty = top_20_shipper[col].apply(format_kmt)
    formatted_pct = percentage_table_shipper[col].fillna("0.0%")
    combined_table_shipper[col] = formatted_qty + '  ' + formatted_pct

# Clean up zeros
for col in month_only:
    combined_table_shipper[col] = combined_table_shipper[col].replace(
        ['0.0k  0.0%', 'nank  0.0%', '0.0k (nan%)'], ''
    )

# === FINAL HTML TABLE ===
combined_table_html = combined_table_shipper.to_html(index=False)


In [0]:

filtered_df_W

# Group by 'Shipper' and sum 'Quantity'
grouped_by_shipper = filtered_df_W.groupby('Shipper')['Quantity'].sum().reset_index()

grouped_by_coord = filtered_df_W.groupby('Coordinator')['Quantity'].sum().reset_index()

total_sum_shipper = grouped_by_shipper['Quantity'].sum()
total_sum_coord = grouped_by_coord['Quantity'].sum()

# Calculate the percentage for each shipper
grouped_by_shipper['Percentage'] = (grouped_by_shipper['Quantity'] / total_sum_shipper) * 100
top_10_shippers = grouped_by_shipper.sort_values('Percentage', ascending=False).head(10)

grouped_by_coord['Percentage'] = (grouped_by_coord['Quantity'] / total_sum_coord) * 100
top_10_coords = grouped_by_coord.sort_values('Percentage', ascending=False).head(10)

# Chart 2: Top 10 Shippers (Pie Chart)
fig_shippers = go.Figure()

fig_shippers.add_trace(go.Pie(
    labels=top_10_shippers['Shipper'],
    values=top_10_shippers['Percentage'],
    textinfo='label+percent',
    texttemplate='%{label}<br>%{value:.2f}%',
    insidetextorientation='radial',
    marker=dict(colors=px.colors.sequential.Blues),
    hovertemplate='%{label}<br>%{value:.2f}%<extra></extra>'
))



fig_shippers.update_layout(
    title="Top 10 Shippers",
    template="plotly_white"
)

fig_shippers.show()

fig_coords = go.Figure()

fig_coords.add_trace(go.Pie(
    labels=top_10_coords['Coordinator'],
    values=top_10_coords['Percentage'],
    textinfo='label+percent',
    texttemplate='%{label}<br>%{value:.2f}%',
    insidetextorientation='radial',
    marker=dict(colors=px.colors.sequential.Blues),
    hovertemplate='%{label}<br>%{value:.2f}%<extra></extra>'
))


fig_coords.update_layout(
    title="Top 10 Coordinators",
    template="plotly_white"
)



fig_coords.show()

fig_shippers_html = fig_shippers.to_html(include_plotlyjs='cdn', full_html=True)
fig_coords_html = fig_coords.to_html(include_plotlyjs='cdn', full_html=True)

## PORTS

In [0]:
current_season=selected_season
df_current_season=history[history['Season'] == current_season]

port_zone_map = {
    'Necochea': 'Neco',
    'Bahia Blanca': 'BB'
}

# Default all others to 'UPR'
df_current_season['zone'] = df_current_season['Port'].apply(lambda x: port_zone_map.get(x, 'UPR'))

In [0]:
import plotly.graph_objects as go
from datetime import datetime

# Make sure Marketing_Month_Name exists
df_current_season['Marketing_Month'] = df_current_season['Date'].dt.month

df_current_season['Marketing_Month_Name'] = df_current_season['Marketing_Month'].map(month_map)

# Group by Month and zone
grouped = df_current_season.groupby(['Marketing_Month_Name', 'zone'])['Quantity'].sum().reset_index()

# Ensure month order is preserved
grouped['Marketing_Month_Name'] = pd.Categorical(grouped['Marketing_Month_Name'], categories=month_names, ordered=True)
grouped = grouped.sort_values(by='Marketing_Month_Name')

# Get current month name in marketing year
current_month_number = datetime.now().month
current_marketing_month = month_map[current_month_number]

# Calculate monthly total for % share
monthly_totals = grouped.groupby('Marketing_Month_Name')['Quantity'].transform('sum')
grouped['Pct'] = (grouped['Quantity'] / monthly_totals) * 100

# Format labels: e.g. "23 kmt\n(45%)"
def format_label(x, pct):
    return f"{int(x / 1000):,} kmt<br>({pct:.0f}%)"

# Build figure
fig = go.Figure()
zones = ['Neco', 'BB', 'UPR']
colors = {'Neco': 'orange', 'BB': 'blue', 'UPR': 'green'}

for zone in zones:
    zone_data = grouped[grouped['zone'] == zone]
    fig.add_trace(go.Bar(
        x=zone_data['Marketing_Month_Name'],
        y=zone_data['Quantity'],
        name=zone,
        marker_color=colors[zone],
        text=[format_label(q, p) for q, p in zip(zone_data['Quantity'], zone_data['Pct'])],
        textposition='outside',
        textfont=dict(size=400),
        hovertemplate=(
        f"<b>{zone}</b><br>"
        "Month: %{x}<br>"
        "Quantity: %{y:,.0f} mt<br>"
        "<extra></extra>"  # Removes the trace name box
    )
    ))

# Layout settings
fig.update_layout(
    barmode='group',
    title='ARG Exports by Port',
    xaxis_title='Month',
    yaxis_title='Quantity (tons)',
    template='plotly_white',
    legend_title='Zone',
    height=700,
    width=950,
    hovermode='x unified'
      # Base font (titles, axes, etc.)
)
fig.update_traces(textfont_size=40)

fig.show()

fig_html = fig.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
import calendar
current_month = calendar.month_name[selected_month]


In [0]:
barley_html= f"""
<!DOCTYPE html>
<html>
<head>
    <title>ARG WHEAT LINE UP REPORT</title>
    <style>
body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}

        h1, h2, h3 {{
            color: #2c3e50;
        }}

        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
            margin-bottom: 30px;
            font-size: 14px;
        }}

        th, td {{
            border: 1px solid #ccc;
            padding: 8px 12px;
            text-align: left;
        }}

        th {{
            background-color: #f2f2f2;
            color: #333;
            font-weight: bold;
        }}

        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}

        tr:hover {{
            background-color: #f1f1f1;
        }}

        .layout {{
            width: 100%;
        }}

        .table-container, .extra-container {{
            vertical-align: top;
            padding: 10px;
        }}

        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
      <h1>{current_month} Exports</h1>
     
        <h2>Exports Overview</h2>
            <div class="chart-container">{overview_html}</div>
        <h2>Current week numbers</h2>
            {summary_df_html}  <!-- Insert DataFrame as an HTML table -->
             
      <h1>{current_month} Exports by Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Destinations</h2>
                  <div class="chart-container">{bar_html}</div>
              </td>
              <!-- Table Section -->
              <td class="table-container">
                <h2>{current_month} Line Ups</h2>
                  {merged_with_subtotals_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
              
          </tr>
      </table>
      <h1>{current_season} Top 10 Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Season {current_season} Main Destinations</h2>
                  <div class="chart-container">{top_html}</div>
              </td>
              <!-- Table Section -->
              <td class="table-container">
                  <h2>Accumulated 2025 exports of main destinations</h2>
                  {top_10_countries_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
          </tr>
      </table>
      <h1>Top 10 players</h1>
      <h2>Top 10 Shippers</h2>
        <div class="chart-container">{fig_shippers_html}</div>
        {combined_table_html}

        

      <h2>Top 10 Coordinators</h2>
        <div class="chart-container">{fig_coords_html}</div>
        {combined_table_coord_html}

      <h1>Top Ports</h1>

        <div class="chart-container">{fig_html}</div>

  </body>
  </html>
  """

barley_report_bytes = barley_html.encode("utf-8")

In [0]:
html_content = f"""
  <!DOCTYPE html>
  <html>
  <head>
      <title>WHEAT Lineups Overview </title>
      <style>
          .container {{
              justify-content: space-between;
              align-items: flex-start;
              width: 100%;
              gap: 10px;
          }}
          .chart-container, .table-container, .extra-container {{
              flex: 0;
              padding: 10px;
              height: 100%;
          }}
          .table-container {{
              text-align: center;
              margin: auto;
              flex: 1; 
              min-width: 300px; 
          }}
          table {{
              width: 60%;
              border-collapse: collapse;
          }}
          th, td {{
              border: 1px solid black;
              padding: 8px;
              text-align: center;
          }}
          th {{
              background-color: #f2f2f2;
          }}
          img {{
              width: 100%;
              height: auto;
              min-width: 750px; /* Set a minimum width */
              min-height: 750px; 
              object-fit: contain;
          }}
          .wide-table {{
              width: 100%;
              table-layout: fixed;
          }}

          .wide-table th, .wide-table td {{
              padding: 8px;
              width: 40%;
              min-width: 100px;
              white-space: nowrap;
          }}
          
      </style>
  </head>
  <body>
      <h1>{current_month} {current_season} Exports</h1>
     
        <h2>Exports Overview</h2>
            <img src="chart1" alt="Barley Exports overview">
        <h2>Current week numbers</h2>
            {summary_df_html}  <!-- Insert DataFrame as an HTML table -->
             
      <h1>{current_month} Exports by Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Destinations</h2>
                  <img src="chart2" alt="Board Crush Chart">
              </td>
              <!-- Table Section -->
              <td class="table-container">
                <h2>{current_month} Line Ups</h2>
                  {merged_with_subtotals_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
              
          </tr>
      </table>
      <h1>{current_season} Top 10 Destinations</h1>
      <table class="layout">
          <tr>
              <!-- New Container Section (before the chart) -->
              <td class="extra-container">
                  <h2>Season {current_season} Main Destinations</h2>
                  <img src="chart3" alt="top 10">
              </td>
              <!-- Table Section -->
              <td class="table-container">
                  <h2>Accumulated 2025 exports of main destinations</h2>
                  {top_10_countries_html}  <!-- Insert DataFrame as an HTML table -->
              </td>
          </tr>
      </table>
      <h1>Top 10 players</h1>
      <h2>Top 10 Shippers</h2>
        <img src="chart4" alt="Top Shippers">
        {combined_table_html}        

      <h2>Top 10 Coordinators</h2>
        <img src="chart5" alt="Top Coordinators">
        {combined_table_coord_html}

      <h1>Top Ports</h1>

        <img src="chart6" alt="Top ports">

  </body>
  </html>
  """

grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com','nicolas.benaicha@ldc.com']

moi=['florian.girardi-ext@ldc.com']

# # Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=email_to_send,
    subject=f'Wheat Lineups {current_month} {current_season}',
    from_addr="florian.girardi-ext@ldc.com",
    body=html_content,
    mime_type="html",
    html_images={"chart1": overview,"chart2":bar,'chart3':top,'chart4':fig_shippers,'chart5':fig_coords,'chart6':fig},
    attachment={'wheat_arg_lineup.html':barley_report_bytes}  # Embedding the chart image
)

